In [5]:
import pandas as pd
import numpy as np

In [9]:
# Load the raw CSV file
df_raw = pd.read_csv("Data/Interview LegalWriter_June 13, 2026_08.27.csv")

# Filter out the first two metadata/header rows from Qualtrics
df_data = df_raw.iloc[2:].reset_index(drop=True)

# Clean domain column (Q2) to ensure no leading/trailing spaces interfere with grouping
df_data['Q2'] = df_data['Q2'].astype(str).str.strip()

# Define criteria sub-indices (odd column index = Brief A, even column index = Brief B)
criteria_sub_indices = {
    "Juridische Accuratesse": (1, 2),
    "Feitelijke Juistheid": (3, 4),
    "Contextuele Integriteit & Logica": (5, 6),
    "Linguïstische Getrouwheid": (7, 8),
    "Actionable Usability": (9, 10)
}

# Mapping schema to align Brief A/B to Model/Baseline per case
case_configurations = {
    "Zaak 1": {"prefix": "QID1_", "model_idx": 2, "baseline_idx": 1},  # A = Baseline, B = Model
    "Zaak 2": {"prefix": "QID6_", "model_idx": 1, "baseline_idx": 2},  # A = Model, B = Baseline
    "Zaak 3": {"prefix": "Zaak 3_", "model_idx": 2, "baseline_idx": 1} # A = Baseline, B = Model
}

# Cast all relevant evaluation columns to float for calculation
for case, config in case_configurations.items():
    pfx = config["prefix"]
    for i in range(1, 11):
        df_data[f"{pfx}{i}"] = pd.to_numeric(df_data[f"{pfx}{i}"], errors='coerce')

print(f"Data prepared. Found {df_data['Q2'].nunique()} unique domain(s) in the dataset.")

Data prepared. Found 2 unique domain(s) in the dataset.


In [10]:
# List to hold processed score rows
domain_results = []

# Iterate through each unique domain found in column Q2
for domain in df_data['Q2'].unique():
    # Filter dataset for the current domain context
    df_domain = df_data[df_data['Q2'] == domain]
    
    for criterion, (idx_a, idx_b) in criteria_sub_indices.items():
        model_values = []
        baseline_values = []
        
        for case, config in case_configurations.items():
            pfx = config["prefix"]
            
            # Map column names dynamically based on the case model setup
            m_col = f"{pfx}{idx_b}" if config["model_idx"] == 2 else f"{pfx}{idx_a}"
            b_col = f"{pfx}{idx_a}" if config["baseline_idx"] == 1 else f"{pfx}{idx_b}"
            
            # Gather valid numeric scores within this domain
            model_values.extend(df_domain[m_col].dropna().tolist())
            baseline_values.extend(df_domain[b_col].dropna().tolist())
            
        # Calculate average ratings per domain context
        mean_model = np.mean(model_values) if model_values else np.nan
        mean_baseline = np.mean(baseline_values) if baseline_values else np.nan
        
        domain_results.append({
            "Domein": domain,
            "Evaluatie Criterium": criterion,
            "Huidig Model (Gemiddelde)": round(mean_model, 2) if not np.isnan(mean_model) else "N/A",
            "Baseline (Gemiddelde)": round(mean_baseline, 2) if not np.isnan(mean_baseline) else "N/A"
        })

# Display the aggregated metrics grouped by Domain
df_domain_scores = pd.DataFrame(domain_results)
df_domain_scores.sort_values(by=["Domein", "Evaluatie Criterium"]).reset_index(drop=True)

,Domein,Evaluatie Criterium,Huidig Model (Gemiddelde),Baseline (Gemiddelde)
0,1,Actionable Usability,N/A,N/A
1,1,Contextuele Integriteit & Logica,N/A,N/A
2,1,Feitelijke Juistheid,N/A,N/A
3,1,Juridische Accuratesse,N/A,N/A
4,1,Linguïstische Getrouwheid,N/A,N/A
5,2,Actionable Usability,3.5,3.17
6,2,Contextuele Integriteit & Logica,4.17,3.33
7,2,Feitelijke Juistheid,4.67,4.17
8,2,Juridische Accuratesse,4.67,3.33
9,2,Linguïstische Getrouwheid,4.5,3.83


In [ ]:
# Structural definition of open text fields across cases
feedback_fields = {
    "Zaak 1": {"Voorkeur": "QID13", "Waarom": "QID18", "Correctheid": "QID20", "Stijl": "QID19", "Fouten": "QID12"},
    "Zaak 2": {"Voorkeur": "QID14", "Waarom": "QID21", "Correctheid": "QID22", "Stijl": "QID23", "Fouten": "QID16"},
    "Zaak 3": {"Voorkeur": "QID15", "Waarom": "Q24", "Correctheid": "QID25", "Stijl": "QID26", "Fouten": "QID17"}
}

# Print expert feedback segmented cleanly by domain group
for domain in sorted(df_data['Q2'].unique()):
    print("=" * 80)
    print(f" DOMEIN GROEP: {domain}")
    print("=" * 80)
    
    df_domain = df_data[df_data['Q2'] == domain]
    
    for case_name, cols in feedback_fields.items():
        print(f"\n--- {case_name} ---")
        
        for label, col_name in cols.items():
            # Clean and filter textual strings
            text_series = df_domain[col_name].dropna().astype(str).str.strip()
            valid_comments = [msg for msg in text_series if msg not in ["", "-", "NaN", "nan", "None"]]
            
            print(f" ➔ {label}:")
            if valid_comments:
                for idx, comment in enumerate(valid_comments, 1):
                    print(f"    [{idx}] {comment}")
            else:
                print("    Geen specifieke feedback ingevoerd door respondenten in dit domein.")
    print("\n" + "#" * 80 + "\n")

 DOMEIN GROEP: 1

--- Zaak 1 ---
 ➔ Voorkeur:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Waarom:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Correctheid:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Stijl:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Fouten:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.

--- Zaak 2 ---
 ➔ Voorkeur:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Waarom:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Correctheid:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Stijl:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Fouten:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.

--- Zaak 3 ---
 ➔ Voorkeur:
    Geen specifieke feedback ingevoerd door respondenten in dit domein.
 ➔ Waaro

: 